In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
from ml_experiments.analyze import get_df_runs_from_mlflow_sql, get_missing_entries, get_common_combinations, get_df_with_combinations
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
import os
import pickle

# Save Results

## Load mlflow runs

In [ ]:
results_dir = Path.cwd().parent / "results" / "spheres"
os.makedirs(results_dir, exist_ok=True)

In [ ]:
db_port = 5001
db_name = 'cohirf'
url = f'postgresql://user@localhost:{db_port}/{db_name}'
engine = create_engine(url)
query = 'SELECT experiments.name from experiments'
experiment_names = pd.read_sql(query, engine)['name'].tolist()

In [ ]:
experiment_names

In [ ]:
experiments_names = [exp for exp in experiment_names if exp.startswith("sphere-")]

In [ ]:
experiments_names

In [ ]:
params_columns = [
    "model",
    "seed_model",
    "seed_dataset",
    "n_samples",
    "n_spheres",
    "radius_separation",
    "radius_std",
    "add_radius_as_feature",
    "best/child_run_id",
	"model_params/n_batches"
]

In [ ]:
latest_metrics_columns = [
    'fit_model_return_elapsed_time',
    'max_memory_used',
    'n_clusters_',
    'rand_score',
    'adjusted_rand',
    'mutual_info',
    'adjusted_mutual_info',
    'normalized_mutual_info',
    'homogeneity',
    'completeness',
    'v_measure',
    'silhouette',
    'calinski_harabasz_score',
    'davies_bouldin_score',
    'inertia_score',
    'best/n_clusters_',
    'best/rand_score',
    'best/adjusted_rand',
    'best/mutual_info',
    'best/adjusted_mutual_info',
    'best/normalized_mutual_info',
    'best/homogeneity_completeness_v_measure',
    'best/silhouette',
    'best/calinski_harabasz_score',
    'best/davies_bouldin_score',
    'best/inertia_score',
    'best/homogeneity',
    'best/completeness',
    'best/v_measure',
	'best/elapsed_time',
]

In [ ]:
tags_columns = [
    'raised_exception',
    'EXCEPTION',
    'mlflow.parentRunId',
]

In [ ]:
runs_columns = ['run_uuid', 'status', 'start_time', 'end_time']
experiments_columns = []
other_table = 'params'
other_table_keys = params_columns
df_params = get_df_runs_from_mlflow_sql(engine, runs_columns=runs_columns, experiments_columns=experiments_columns, experiments_names=experiments_names, other_table=other_table, other_table_keys=other_table_keys)
df_latest_metrics = get_df_runs_from_mlflow_sql(engine, runs_columns=['run_uuid'], experiments_columns=experiments_columns, experiments_names=experiments_names, other_table='latest_metrics', other_table_keys=latest_metrics_columns)
df_tags = get_df_runs_from_mlflow_sql(engine, runs_columns=['run_uuid'], experiments_columns=experiments_columns, experiments_names=experiments_names, other_table='tags', other_table_keys=tags_columns)

In [ ]:
df_runs_raw = df_params.join(df_latest_metrics)
df_runs_raw = df_runs_raw.join(df_tags)
df_runs_raw.to_csv(results_dir / 'df_runs_raw.csv', index=True)

In [ ]:
df_runs_raw = pd.read_csv(results_dir / "df_runs_raw.csv", index_col=0)
df_runs_raw_parents = df_runs_raw.copy()
df_runs_raw_parents = df_runs_raw_parents.loc[df_runs_raw_parents['mlflow.parentRunId'].isna()]

In [ ]:
df_runs_raw_parents

## Delete duplicate runs (if any) and complete some models that cannot run with some datasets

In [ ]:
non_duplicate_columns = [
    "model",
    "seed_model",
    "seed_dataset",
    "n_samples",
    "n_spheres",
    "radius_separation",
    "radius_std",
    "add_radius_as_feature",
]
df_runs_parents = df_runs_raw_parents.dropna(axis=0, how="all", subset=["best/adjusted_rand"]).copy()
df_runs_parents = df_runs_parents.loc[(~df_runs_parents.duplicated(non_duplicate_columns))]
# fill missing values with "None"
df_runs_parents = df_runs_parents.fillna("None")

In [ ]:
df = df_runs_parents.copy()
df = df.loc[df['model'] == 'BatchCoHiRF-DBSCAN-1iter']
df = df[["n_samples", "model_params/n_batches"]]

In [ ]:
df

# Missing

In [ ]:
model_nickname = df_runs_parents['model'].unique().tolist()
model_nickname.sort()
model_nickname

In [ ]:
model_nickname = [
    # 'BatchCoHiRF-DBSCAN-1iter', 
	# 'CoHiRF-DBSCAN', 
	'DBSCAN'
]

In [ ]:
non_duplicate_columns = [
    "model",
    # "seed_model",
    "n_samples",
    "seed_dataset",
    # "n_spheres",
    # "radius_separation",
    # "radius_std",
    # "add_radius_as_feature",
    
]

In [ ]:
n_samples = [200, 2000, 20000, 200000] #2000000, 20000000]  # , 200000]
seed_dataset = [i for i in range(5)]
columns_names = non_duplicate_columns
should_contain_values = [model_nickname, n_samples, seed_dataset]
df_missing = get_missing_entries(df_runs_parents, columns_names, should_contain_values)

In [ ]:
df_missing

In [ ]:
df_missing["hpo_seed"] = df_missing["seed_dataset"]

In [ ]:
# Join df_runs_raw_parents into df_missing using non_duplicate_columns to get the EXCEPTION column
df_missing_with_exception = df_missing.merge(
    df_runs_raw_parents[non_duplicate_columns + ["raised_exception"]],
    how="left",
    left_on=["model", "n_samples", "seed_dataset"],
    right_on=["model", "n_samples", "seed_dataset"],
)
df_missing_with_exception[["model", "n_samples", "seed_dataset", "raised_exception"]]

In [ ]:
missing_dict = {}
for model in df_missing['model'].unique():
    sub = df_missing[df_missing["model"] == model].drop(columns=["model"])
    missing_dict[model] = sub.to_dict(orient="records")
if len(missing_dict) != 0:
    with open(results_dir / 'missing_dict.pkl', 'wb') as f:
        pickle.dump(missing_dict, f)

# Get common combinations

In [ ]:
model_nickname = [
    'BatchCoHiRF-DBSCAN-1iter', 
	# 'BatchCoHiRF-DBSCAN',
	'CoHiRF-DBSCAN', 
	'DBSCAN'
]
df = df_runs_parents.copy()
df = df.loc[df["model"].isin(model_nickname)]
column = 'model'
combination_columns = [
    "n_samples",
    "seed_dataset",
]
common_combinations = get_common_combinations(df, column, combination_columns)

In [ ]:
df_common = get_df_with_combinations(df, combination_columns, common_combinations)

In [ ]:
model_nickname = [
    'BatchCoHiRF-DBSCAN-1iter', 
	# 'BatchCoHiRF-DBSCAN',
]
df = df_runs_parents.copy()
df = df.loc[df["model"].isin(model_nickname)]
df = df.loc[df["n_samples"] == 200000]
column = 'model'
combination_columns = [
    "n_samples",
    "seed_dataset",
]
common_combinations = get_common_combinations(df, column, combination_columns)
df_common2 = get_df_with_combinations(df, combination_columns, common_combinations)

In [ ]:
df_common = pd.concat([df_common, df_common2], ignore_index=True)

In [ ]:
df = df_runs_parents.copy()
df = df.loc[df["model"] == 'CoHiRF-DBSCAN']
df = df.loc[df["n_samples"] == 2000]
df

# Plots

In [ ]:
df = df_runs_parents.copy()
models_names = {
    "BatchCoHiRF-DBSCAN-1iter": "BatchCoHiRF-DBSCAN",
    "DBSCAN": "DBSCAN",
}
df = df.loc[df["model"].isin(models_names.keys())]
df["n_samples"] = df["n_samples"].astype(int)
df = df.loc[df["n_samples"] >= 2000]
df = df.replace({"model": models_names})
df = df.sort_values(by="model")
df = df.rename(
    columns={
        "fit_model_return_elapsed_time": "Time (s)",
        "max_memory_used": "Memory (MB)",
        "n_samples": "Number of samples",
        "model": "Model",
        "best/adjusted_rand": "ARI",
    }
)

with mpl.rc_context(
    rc={
        "font.family": "serif",
        "font.serif": ["DejaVu Serif", "Liberation Serif", "Times", "serif"],  # Reliable serif fonts
        "mathtext.fontset": "dejavuserif",  # Use DejaVu for math text to avoid missing glyphs
        "axes.unicode_minus": False,  # Use ASCII minus instead of Unicode minus
        "font.size": 12,
        "axes.linewidth": 1.2,
        "axes.labelsize": 16,
        "axes.titlesize": 16,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "legend.fontsize": 10,
        "legend.frameon": True,
        "legend.fancybox": False,
        "legend.shadow": False,
        "legend.framealpha": 0.9,
        "legend.edgecolor": "black",
        "grid.alpha": 0.3,
        "axes.grid": True,
        "grid.linewidth": 0.5,
    }
):
    cm = 1 / 2.54  # centimeters to inches
    scale = 3.0
    fig = plt.figure(figsize=(12 * scale * cm, 7 * scale * cm))
    palette = sns.color_palette("tab10", n_colors=len(df["Model"].unique()))
    # ARI plot
    ax = sns.lineplot(
        data=df,
        x="Number of samples",
        y="ARI",
        hue="Model",
        style="Model",
        markers=True,
        markersize=10,
        errorbar=("ci", 95),
        palette=palette,
        legend=True,
        # err_style="bars",
    )
    # ax.set_xscale("log", base=10)
    ax.set_xscale("log")
    ax.set_ylabel("Adjusted Rand Index (ARI)", fontweight='bold')
    ax.set_xlabel("Number of samples", fontweight='bold')
    ax.legend(title='', loc='upper center', ncol=4, fontsize=13, frameon=True, bbox_to_anchor=(0.5, 1.08))
    ax.tick_params(direction="in", length=4, width=1.2)
    for spine in ["left", "right", "top", "bottom"]:
        ax.spines[spine].set_linewidth(1.4)
        ax.spines[spine].set_color("black")
    ax.grid(True, alpha=0.7, linestyle="-", linewidth=0.5)
    fig.savefig(results_dir / f"hpo_sphere_ari_cohirf.pdf", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
df = df_runs_parents.copy()
models_names = {
    "BatchCoHiRF-DBSCAN-1iter": "BatchCoHiRF-DBSCAN",
    "DBSCAN": "DBSCAN",
}
df = df.loc[df["model"].isin(models_names.keys())]
df = df.loc[df["n_samples"] >= 2000]
df = df.replace({"model": models_names})
df = df.sort_values(by="model")
df = df.rename(
    columns={
        "best/elapsed_time": "Time (s)",
        "max_memory_used": "Memory (MB)",
        "n_samples": "Number of samples",
        "model": "Model",
        "best/adjusted_rand": "ARI",
    }
)

with mpl.rc_context(
    rc={
        "font.family": "serif",
        "font.serif": ["DejaVu Serif", "Liberation Serif", "Times", "serif"],  # Reliable serif fonts
        "mathtext.fontset": "dejavuserif",  # Use DejaVu for math text to avoid missing glyphs
        "axes.unicode_minus": False,  # Use ASCII minus instead of Unicode minus
        "font.size": 12,
        "axes.linewidth": 1.2,
        "axes.labelsize": 16,
        "axes.titlesize": 16,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "legend.fontsize": 10,
        "legend.frameon": True,
        "legend.fancybox": False,
        "legend.shadow": False,
        "legend.framealpha": 0.9,
        "legend.edgecolor": "black",
        "grid.alpha": 0.3,
        "axes.grid": True,
        "grid.linewidth": 0.5,
    }
):
    cm = 1 / 2.54  # centimeters to inches
    scale = 3.0
    fig, axs = plt.subplots(2, 1, sharex=True, figsize=(12 * scale * cm, 14 * scale * cm))
    axs = axs.flatten()
    palette = sns.color_palette("tab10", n_colors=len(df["Model"].unique()))
    # Time plot
    ax = sns.lineplot(
        data=df,
        x="Number of samples",
        y="Time (s)",
        hue="Model",
        style="Model",
        markers=True,
        markersize=10,
        errorbar=("ci", 95),
        palette=palette,
        legend=True,
        ax=axs[0],
    )
    # ax.set_yscale("log")
    # ARI plot
    ax2 = sns.lineplot(
        data=df,
        x="Number of samples",
        y="ARI",
        hue="Model",
        style="Model",
        markers=True,
        markersize=10,
        errorbar=("ci", 95),
        palette=palette,
        legend=True,
        ax=axs[1],
    )
    ax.set_xscale("log")
    ax2.set_xscale("log")
    ax.set_ylabel("Time (s)", fontweight='bold')
    ax2.set_ylabel("Adjusted Rand Index (ARI)", fontweight='bold')
    ax2.grid(True, which="both", linestyle="--", linewidth=0.5)
    # Remove duplicate legends
    handles, labels = ax.get_legend_handles_labels()
    # fig.legend(handles, labels, loc="upper center", ncol=4, fontsize=13, frameon=False, bbox_to_anchor=(0.5, 1.02))
    # ax.get_legend().remove()
    ax.legend(title="", loc="upper center", ncol=4, fontsize=13, frameon=True, bbox_to_anchor=(0.5, 1.08))
    ax2.get_legend().remove()
    plt.xlabel("Number of samples")
    ax.tick_params(direction="in", length=4, width=1.2)
    ax.grid(True, which="both", linestyle="--", linewidth=0.5)
    ax2.grid(True, which="both", linestyle="--", linewidth=0.5)
    for spine in ["left", "right", "top", "bottom"]:
        ax.spines[spine].set_linewidth(1.4)
        ax.spines[spine].set_color("black")
        ax2.spines[spine].set_linewidth(1.4)
        ax2.spines[spine].set_color("black")
    # Reduce vertical space between subplots
    plt.subplots_adjust(hspace=0.05)
    fig.savefig(results_dir / f"hpo_sphere_ari_cohirf_best_time.pdf", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
df = df_runs_parents.copy()
models_names = {
	"BatchCoHiRF-DBSCAN-1iter": "BatchCoHiRF-DBSCAN",
	# "BatchCoHiRF-DBSCAN": "BatchCoHiRF-DBSCAN",
	# "CoHiRF-DBSCAN": "CoHiRF-DBSCAN",
	"DBSCAN": "DBSCAN",
}
df = df.loc[df["model"].isin(models_names.keys())]
df = df.loc[df["n_samples"] >= 2000]
df = df.replace({"model": models_names})
df = df.sort_values(by="model")
df = df.rename(
	columns={
		"fit_model_return_elapsed_time": "Time (s)",
		"max_memory_used": "Memory (MB)",
		"n_samples": "Number of samples",
		"model": "Model",
		"best/adjusted_rand": "ARI",
	}
)

plt.style.use("seaborn-v0_8-whitegrid")
with mpl.rc_context(
	rc={
		"figure.constrained_layout.use": True,
		"savefig.bbox": "tight",
		"figure.figsize": (12, 7),
		"legend.loc": "upper left",
		"legend.frameon": True,
		"font.size": 14,
		"axes.titlesize": 16,
		"axes.labelsize": 15,
		"xtick.labelsize": 13,
		"ytick.labelsize": 13,
		"axes.grid": True,
		"grid.color": "grey",
		"grid.alpha": 0.3,
	}
):
	fig, axs = plt.subplots(2, 1, sharex=True)
	axs = axs.flatten()
	palette = sns.color_palette("tab10", n_colors=len(df["Model"].unique()))
	# Time plot
	ax = sns.lineplot(
		data=df,
		x="Number of samples",
		y="Time (s)",
		hue="Model",
		style="Model",
		markers=True,
		dashes=False,
		errorbar="ci",
		ax=axs[0],
		palette=palette,
	)
	ax.set_yscale("log")
	ax.set_ylabel("Time (s)")
	ax.grid(True, which="both", linestyle="--", linewidth=0.5)
	# ARI plot
	ax2 = sns.lineplot(
		data=df,
		x="Number of samples",
		y="ARI",
		hue="Model",
		style="Model",
		markers=True,
		dashes=False,
		errorbar="ci",
		ax=axs[1],
		palette=palette,
	)
	ax2.set_xscale("log")
	ax2.set_ylabel("Adjusted Rand Index (ARI)")
	ax2.grid(True, which="both", linestyle="--", linewidth=0.5)
	# Remove duplicate legends
	handles, labels = ax.get_legend_handles_labels()
	fig.legend(handles, labels, loc="upper center", ncol=4, fontsize=13, frameon=False, bbox_to_anchor=(0.5, 1.08))
	ax.get_legend().remove()
	ax2.get_legend().remove()
	plt.xlabel("Number of samples")
	plt.tight_layout()
	plt.savefig(
	    results_dir / f"hpo_sphere_ari_cohirf.pdf", dpi=300
	)
	plt.show()

In [ ]:
df = df_runs_parents.copy()
models_names = {
    "BatchCoHiRF-DBSCAN-1iter": "BatchCoHiRF-DBSCAN",
    "DBSCAN": "DBSCAN",
}
df = df.loc[df["model"].isin(models_names.keys())]
df = df.replace({"model": models_names})
df = df.sort_values(by="model")
df = df.rename(
    columns={
        "best/elapsed_time": "Time (s)",
        "max_memory_used": "Memory (MB)",
        "n_samples": "Number of samples",
        "model": "Model",
        "best/adjusted_rand": "ARI",
    }
)

plt.style.use("seaborn-v0_8-whitegrid")
with mpl.rc_context(
    rc={
        "figure.constrained_layout.use": True,
        "savefig.bbox": "tight",
        "figure.figsize": (12, 7),
        "legend.loc": "upper left",
        "legend.frameon": True,
        "font.size": 14,
        "axes.titlesize": 16,
        "axes.labelsize": 15,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "axes.grid": True,
        "grid.color": "grey",
        "grid.alpha": 0.3,
    }
):
    fig, axs = plt.subplots(2, 1, sharex=True)
    axs = axs.flatten()
    palette = sns.color_palette("tab10", n_colors=len(df["Model"].unique()))
    # Time plot
    ax = sns.lineplot(
        data=df,
        x="Number of samples",
        y="Time (s)",
        hue="Model",
        style="Model",
        markers=True,
        dashes=False,
        errorbar="ci",
        ax=axs[0],
        palette=palette,
    )
    ax.set_yscale("log")
    ax.set_ylabel("Time (s)")
    ax.grid(True, which="both", linestyle="--", linewidth=0.5)
    # ARI plot
    ax2 = sns.lineplot(
        data=df,
        x="Number of samples",
        y="ARI",
        hue="Model",
        style="Model",
        markers=True,
        dashes=False,
        errorbar="ci",
        ax=axs[1],
        palette=palette,
    )
    ax2.set_xscale("log")
    ax2.set_ylabel("Adjusted Rand Index (ARI)")
    ax2.grid(True, which="both", linestyle="--", linewidth=0.5)
    # Remove duplicate legends
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=4, fontsize=13, frameon=False, bbox_to_anchor=(0.5, 1.08))
    ax.get_legend().remove()
    ax2.get_legend().remove()
    plt.xlabel("Number of samples")
    plt.tight_layout()
    plt.savefig(results_dir / f"best_hpo_sphere_ari_cohirf.pdf", dpi=300)
    plt.show()